# Q1 cross-model check (TriadRx, design 9.1, preregistration 5.5.2)

Runtime: GPU T4. Upload `q1_judge_input.jsonl` (from `scripts/quality/05_judge_input.py`) and `q1_judge_v1.txt` (from `prompts/`).
The input contains no human labels. Greedy decoding, thinking disabled.


In [ ]:
!pip install -q -U transformers accelerate bitsandbytes huggingface_hub


In [ ]:
from google.colab import files

# Upload q1_judge_input.jsonl and q1_judge_v1.txt
uploaded = files.upload()


In [ ]:
import datetime
import hashlib
import json
import platform

import torch
import transformers
from huggingface_hub import model_info
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3.5-9B"
# Fallback if the primary model cannot be loaded or run on a T4 (preregistration 5.5.2)
FALLBACK_ID = "Qwen/Qwen3-8B"
INPUT = "q1_judge_input.jsonl"
PROMPT = "q1_judge_v1.txt"
MAX_NEW_TOKENS = 256

def sha256(path):
    with open(path, "rb") as handle:
        return hashlib.sha256(handle.read()).hexdigest()

records = [json.loads(line) for line in open(INPUT, encoding="utf-8") if line.strip()]
system_prompt = open(PROMPT, encoding="utf-8").read()
print(len(records), "records", sha256(INPUT))


In [ ]:
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)

def load(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=quant, device_map="auto", torch_dtype=torch.float16)
    model.eval()
    return tokenizer, model

used_id, load_error = MODEL_ID, None
try:
    tokenizer, model = load(MODEL_ID)
except Exception as exc:
    # Record why the primary model failed, then use the fallback
    load_error = f"{type(exc).__name__}: {exc}"[:500]
    print("primary model failed:", load_error)
    used_id = FALLBACK_ID
    tokenizer, model = load(FALLBACK_ID)
revision = model_info(used_id).sha
print("using", used_id, revision, torch.cuda.get_device_name(0))


In [ ]:
import re

def judge(user_text):
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_text}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    found = re.findall(r"\{.*?\}", raw, flags=re.DOTALL)
    label, reason = None, None
    if found:
        try:
            obj = json.loads(found[-1])
            if obj.get("label") in (0, 1):
                label, reason = obj["label"], obj.get("reason")
        except json.JSONDecodeError:
            pass
    return raw, label, reason, int(inputs["input_ids"].shape[1])

started = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
results = []
for i, rec in enumerate(records):
    raw, label, reason, n_tokens = judge(rec["user"])
    results.append({"review_id": rec["review_id"], "label": label, "reason": reason, "valid": label is not None,
                    "input_tokens": n_tokens, "raw": raw})
    if (i + 1) % 10 == 0:
        print(i + 1, "done")
finished = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
print(sum(r["valid"] for r in results), "valid of", len(results))


In [ ]:
meta = {"model_id": used_id, "model_revision": revision, "primary_model": MODEL_ID, "primary_load_error": load_error,
        "quantization": "bitsandbytes 4-bit nf4, compute float16", "decoding": "greedy, max_new_tokens 256, thinking disabled",
        "input_sha256": sha256(INPUT), "prompt_sha256": sha256(PROMPT), "started": started, "finished": finished,
        "transformers": transformers.__version__, "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0),
        "python": platform.python_version()}
with open("q1_judge_output.jsonl", "w", encoding="utf-8") as handle:
    for r in results:
        handle.write(json.dumps(r, ensure_ascii=False) + "\n")
with open("q1_judge_meta.json", "w") as handle:
    json.dump(meta, handle, indent=2)
files.download("q1_judge_output.jsonl")
files.download("q1_judge_meta.json")
